# Paso 1 — Preprocesamiento de datos (GNN-GO)
Este notebook carga y estandariza los archivos de entrada (`Edge.csv`, `GO.csv`, `metadata_GO.csv`, `metadata_proteins.csv`), construye las **features de nodos** (one-hot de GO + metadata) y las **aristas** con pesos (`interaction_score`), y deja todo listo para PyTorch Geometric:

- `x`: matriz de características de nodos 
- `edge_index`: aristas bidireccionales 
- `edge_attr`: pesos de aristas 

In [1]:
import os
import pandas as pd
import torch
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder, StandardScaler
from collections import defaultdict
from torch_geometric.data import Data

In [2]:
# --- Variables de Configuración Global ---
# Este filtro se usa en create_node_features
GO_ONTOLOGY_FILTER = 'all' 
# El valor 'all' incluye BP (Biological Process), MF (Molecular Function) y CC (Cellular Component)

## Rutas de datos

In [3]:
BASE_INPUT_DIR = './input' 

EDGE_FILENAME = "Edge.csv"
GO_FILENAME = "GO.csv"
PROTEIN_METADATA_FILENAME = "metadata_proteins.csv"
GO_METADATA_FILENAME = "metadata_GO.csv"

edge_path = os.path.join(BASE_INPUT_DIR, EDGE_FILENAME)
go_path = os.path.join(BASE_INPUT_DIR, GO_FILENAME)
protein_metadata_path = os.path.join(BASE_INPUT_DIR, PROTEIN_METADATA_FILENAME)
go_metadata_path = os.path.join(BASE_INPUT_DIR, GO_METADATA_FILENAME)


print("\n--- Verificación de Archivos de Datos ---")
if not os.path.isdir(BASE_INPUT_DIR):
    print(f"🚨 Error: El directorio de entrada '{BASE_INPUT_DIR}' no existe.")
    try:
        os.makedirs(BASE_INPUT_DIR, exist_ok=True)
        print(f"Carpeta '{BASE_INPUT_DIR}' creada. Por favor, coloca los archivos CSV dentro.")
    except Exception as e:
        print(f"No se pudo crear el directorio. Error: {e}")
    raise FileNotFoundError("Directorio de entrada no encontrado. ¡Por favor, revisa la ruta!")


EXPECTED_FILES = {
    "Edge": edge_path,
    "GO": go_path,
    "metadata_proteins": protein_metadata_path, 
    "metadata_GO": go_metadata_path
}

for name, filepath in EXPECTED_FILES.items():
    if not os.path.exists(filepath):
        print(f"🚨 Error: El archivo '{name}' no se encontró en la ruta: {filepath}")
        raise FileNotFoundError(f"Archivo requerido no encontrado: {name}. Asegúrate de que todos los archivos CSV estén en '{BASE_INPUT_DIR}'.")
        
print("✔️ Todos los archivos de datos encontrados en el directorio de entrada.")


--- Verificación de Archivos de Datos ---
✔️ Todos los archivos de datos encontrados en el directorio de entrada.


## load files

In [4]:
def load_files(edge_path, go_path, protein_metadata_path, go_metadata_path):
    edges_df = pd.read_csv(edge_path, sep='\t')
    go_terms_df = pd.read_csv(go_path, sep='\t')
    protein_metadata_df = pd.read_csv(protein_metadata_path, sep=',')
    go_metadata_df = pd.read_csv(go_metadata_path, sep=',')
    return edges_df, go_terms_df, protein_metadata_df, go_metadata_df

print("Cargando datasets CSV...")
edges_df, go_terms_df, protein_metadata_df, go_metadata_df = load_files(
        edge_path, go_path, protein_metadata_path, go_metadata_path)
print("Datasets cargados.")

Cargando datasets CSV...
Datasets cargados.


## create node mappings

In [5]:
def create_node_mappings(edges_df, go_terms_df, protein_metadata_df):
    all_proteins_raw = pd.concat([
        edges_df['proteina1'],
        edges_df['proteina2'], 
        go_terms_df['proteina'],
        protein_metadata_df['proteina']
    ]).unique()
    
    all_proteins = [
        p for p in all_proteins_raw 
        if isinstance(p, str) and not p.startswith('GO:') and pd.notna(p)
    ]

    protein_to_idx = {protein: i for i, protein in enumerate(all_proteins)}
    idx_to_protein = {i: protein for protein, i in protein_to_idx.items()}

    return protein_to_idx, idx_to_protein, all_proteins

print("Creando mapeos de proteínas a índices numéricos...")
protein_to_idx, idx_to_protein, all_proteins = create_node_mappings(edges_df, go_terms_df, protein_metadata_df)

Creando mapeos de proteínas a índices numéricos...


## get go ontology mappings

In [6]:
def get_go_ontology_mapping(go_metadata_df):
    go_ontology_map = {}
    go_terms_by_ontology = {'BP': [], 'MF': [], 'CC': [], 'unknown': []}
    
    for _, row in go_metadata_df.iterrows():
        term = row['GO_term']
        ontology_info = str(row['Ontology']).lower()
        
        if 'biological process' in ontology_info:
            go_ontology_map[term] = 'BP'
            go_terms_by_ontology['BP'].append(term)
        elif 'molecular function' in ontology_info:
            go_ontology_map[term] = 'MF'
            go_terms_by_ontology['MF'].append(term)
        elif 'cellular component' in ontology_info:
            go_ontology_map[term] = 'CC'
            go_terms_by_ontology['CC'].append(term)
        else:
            go_ontology_map[term] = 'unknown'
            go_terms_by_ontology['unknown'].append(term)

    
    go_terms_by_ontology['all'] = (
        go_terms_by_ontology['BP'] + 
        go_terms_by_ontology['MF'] + 
        go_terms_by_ontology['CC']
    )

    return go_ontology_map, go_terms_by_ontology

## create node features

In [7]:
def create_node_features(protein_to_idx, go_terms_df, protein_metadata_df, go_metadata_df, go_ontology_filter='all'):
   
    num_nodes = len(protein_to_idx) 
    
    # 1. Procesamiento de Metadata de Proteínas
    protein_features = pd.DataFrame(index=protein_to_idx.keys())
    protein_features = protein_features.merge(
        protein_metadata_df.set_index('proteina'), 
        left_index=True, right_index=True, how='left')
    
    # Rellenar valores faltantes (Imputación simple)
    protein_features['Target_type'] = protein_features['Target_type'].fillna('unknown')
    protein_features['Target_group'] = protein_features['Target_group'].fillna('')
    protein_features['Target_group_score_normalized'] = protein_features['Target_group_score_normalized'].fillna(0.0)
    protein_features['DEG'] = protein_features['DEG'].fillna('none')

    # Codificación de 'Target_type' y 'DEG' (Label Encoding)
    le_target_type = LabelEncoder()
    protein_features['Target_type_encoded'] = le_target_type.fit_transform(protein_features['Target_type'])
    
    le_deg = LabelEncoder()
    protein_features['DEG_encoded'] = le_deg.fit_transform(protein_features['DEG'])

    # Codificación de 'Target_group' (Multi-Hot Encoding)
    all_target_groups = set()
    for groups in protein_features['Target_group'].dropna():
        for g in str(groups).split(','):
            g = g.strip()
            if g:
                all_target_groups.add(g)
    
    mlb_target_group = MultiLabelBinarizer(classes=sorted(list(all_target_groups)))
    target_group_encoded = mlb_target_group.fit_transform(
        protein_features['Target_group'].apply(lambda x: [g.strip() for g in str(x).split(',') if g.strip()])
    )
    target_group_df = pd.DataFrame(target_group_encoded, index=protein_features.index, columns=mlb_target_group.classes_)

    # Normalizar 'Target_group_score_normalized' (Standard Scaling)
    scaler = StandardScaler()
    protein_features['Target_group_score_normalized_scaled'] = scaler.fit_transform(
        protein_features[['Target_group_score_normalized']]
    )

    # 2. Procesamiento de GO Terms (Multi-Hot Encoding)
    go_ontology_map, go_terms_by_ontology = get_go_ontology_mapping(go_metadata_df)
    valid_go_terms = set(go_terms_by_ontology.get(go_ontology_filter, []))

    filtered_go_terms = go_terms_df[
        (go_terms_df['GO_term'].isin(valid_go_terms)) &
        (go_terms_df['proteina'].isin(protein_to_idx.keys()))
    ]

    protein_go_map = defaultdict(list)
    for _, row in filtered_go_terms.iterrows():
        protein_go_map[row['proteina']].append(row['GO_term'])

    go_terms_for_binarizer = [protein_go_map[p] for p in protein_features.index]

    unique_go_terms = sorted(valid_go_terms.intersection(filtered_go_terms['GO_term'].unique()))
    num_nodes_covered_by_go = len({p for p in filtered_go_terms['proteina']})
    num_go_terms_covered = len(unique_go_terms)

    if unique_go_terms:
        mlb_go = MultiLabelBinarizer(classes=unique_go_terms)
        go_features_encoded = mlb_go.fit_transform(go_terms_for_binarizer)
        go_features_df = pd.DataFrame(go_features_encoded,
                                      index = protein_features.index,
                                      columns=[f"GO_{c}" for c in mlb_go.classes_])
    else:
        mlb_go = MultiLabelBinarizer()
        go_features_df = pd.DataFrame(index=protein_features.index)

    idx_to_go_term_id = {i: go_term for i, go_term in enumerate(mlb_go.classes_)} if unique_go_terms else {}
    
    # Combinar todas las características
    all_features_df = pd.concat([
        protein_features[['Target_type_encoded', 'DEG_encoded', 'Target_group_score_normalized_scaled']],
        target_group_df,
        go_features_df
    ], axis=1)

    # Convertir a tensor de PyTorch (ordenado por protein_to_idx)
    X = torch.tensor(all_features_df.loc[list(protein_to_idx.keys())].values, dtype=torch.float)
    
    return X, num_nodes_covered_by_go, num_go_terms_covered, le_target_type, le_deg, mlb_target_group, mlb_go, idx_to_go_term_id


print(f"Creando características de nodos (features) con filtro GO: '{GO_ONTOLOGY_FILTER}'...")
x, num_nodes_covered_by_go, num_go_terms_covered, le_target_type, le_deg, mlb_target_group, mlb_go, idx_to_go_term_id = create_node_features(
    protein_to_idx,
    go_terms_df,
    protein_metadata_df,
    go_metadata_df,
    go_ontology_filter=GO_ONTOLOGY_FILTER
    )

global IN_CHANNELS
IN_CHANNELS = x.shape[1] 
print(f"Dimensión de las características de nodo (input para GNN): {IN_CHANNELS}")


Creando características de nodos (features) con filtro GO: 'all'...
Dimensión de las características de nodo (input para GNN): 6935


## create edge index and attributes

In [8]:
def create_edge_index_and_attributes(edges_df, protein_to_idx):
   
    filtered_edges = edges_df[
        (edges_df['proteina1'].isin(protein_to_idx.keys())) & 
        (edges_df['proteina2'].isin(protein_to_idx.keys()))
    ].copy() 
    
    src = [protein_to_idx[p] for p in filtered_edges['proteina1']]
    dst = [protein_to_idx[p] for p in filtered_edges['proteina2']]
    
    edge_index = torch.tensor([src + dst, dst + src], dtype=torch.long)
    
    edge_attr = torch.tensor(
        filtered_edges['interaction_score'].values.tolist() * 2,
        dtype=torch.float
    ).unsqueeze(1) 
    
    num_edges_original = len(filtered_edges)
    num_edges_bidirectional = num_edges_original * 2
    
    return edge_index, edge_attr, num_edges_original, num_edges_bidirectional

print("Creando índices de aristas y atributos de aristas (interaction_score)...")
edge_index, edge_attr, num_edges_original, num_edges_bidirectional = create_edge_index_and_attributes(edges_df, protein_to_idx)


Creando índices de aristas y atributos de aristas (interaction_score)...


## Información

In [9]:
print("\n--- Resumen del Grafo Cargado ---")
print(f"  Total de Nodos (Proteínas únicas): {x.shape[0]}")
print(f"  Nodos con GO terms cubiertos por la ontología '{GO_ONTOLOGY_FILTER}': {num_nodes_covered_by_go}")
print(f"  Número de GO terms únicos utilizados (tras filtro '{GO_ONTOLOGY_FILTER}'): {num_go_terms_covered}")
print(f"  Total de Aristas originales (interacciones únicas): {num_edges_original}")
print(f"  Total de Aristas en el grafo (bidireccional): {num_edges_bidirectional}")
print(f"  Dimensión de atributos de arista (`interaction_score`): {edge_attr.shape[1]}")
print(f"  Dimensión de características de nodo: {x.shape[1]}")




--- Resumen del Grafo Cargado ---
  Total de Nodos (Proteínas únicas): 5390
  Nodos con GO terms cubiertos por la ontología 'all': 5161
  Número de GO terms únicos utilizados (tras filtro 'all'): 6928
  Total de Aristas originales (interacciones únicas): 430206
  Total de Aristas en el grafo (bidireccional): 860412
  Dimensión de atributos de arista (`interaction_score`): 1
  Dimensión de características de nodo: 6935


# Guardar 'data'

In [10]:
print("Creando objeto Data de PyTorch Geometric...")
data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

Creando objeto Data de PyTorch Geometric...


In [11]:
BASE_OUTPUT_DIR = './output' 
if not os.path.exists(BASE_OUTPUT_DIR):
    os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
    print(f"\nDirectorio de salida '{BASE_OUTPUT_DIR}' creado.")


output_path = os.path.join(BASE_OUTPUT_DIR, 'processed_graph_data.pt')
torch.save(data, output_path)
print(f"Objeto 'data' guardado en: {output_path}")

metadata = {
    'protein_to_idx': protein_to_idx,
    'idx_to_protein': idx_to_protein,
    'idx_to_go_term_id': idx_to_go_term_id,
    'encoders': {
        'le_target_type': le_target_type,
        'le_deg': le_deg,
        'mlb_target_group': mlb_target_group,
        'mlb_go': mlb_go
    }
}
metadata_path = os.path.join(BASE_OUTPUT_DIR, 'metadata.pt')
torch.save(metadata, metadata_path)
print(f"Metadata (mapeos y codificadores) guardada en: {metadata_path}")

print("\n✔️ Archivos guardados correctamente. Puedes continuar con 02_tuning.ipynb.")



Objeto 'data' guardado en: ./output/processed_graph_data.pt
Metadata (mapeos y codificadores) guardada en: ./output/metadata.pt

✔️ Archivos guardados correctamente. Puedes continuar con 02_tuning.ipynb.
